In [3]:
#packages 
import pandas as pd

In [4]:
# Load both sources with matching text identifiers.
imdb = pd.read_csv("data/imdb_movies.csv", dtype={"imdb_id": "string", "nconst": "string"})
tmdb = pd.read_csv("data/tmdb_movies.csv", dtype={"imdb_id": "string"})
for dataset in (tmdb, imdb):
    dataset["imdb_id"] = dataset["imdb_id"].str.strip().replace("", pd.NA)


In [5]:
# Multiple IMDb rows per movie are expected when it has co-directors.
print("Missing movie IDs (TMDB, IMDb):", tmdb["imdb_id"].isna().sum(), imdb["imdb_id"].isna().sum())
print("Repeated movie IDs (TMDB, IMDb):", tmdb["imdb_id"].duplicated().sum(), imdb["imdb_id"].duplicated().sum())


Missing movie IDs (TMDB, IMDb): 10 0
Repeated movie IDs (TMDB, IMDb): 9 84483


Each table has 10 rows with a missing `imdb_id` and 9 duplicates. The 9 "duplicates" in each are most likely just  blanks being counted as repeats of each other

In [6]:
# An inner join needs movie IDs in both sources.
tmdb = tmdb.dropna(subset=["imdb_id"])
imdb = imdb.dropna(subset=["imdb_id", "nconst"])
imdb = imdb.drop_duplicates(subset=["imdb_id", "nconst"])
if not tmdb["imdb_id"].is_unique:
    raise ValueError("TMDB has repeated IMDb movie IDs; resolve them before joining.")
print("Unique movie–director pairs:", len(imdb))


Unique movie–director pairs: 763387


In [20]:
# Keep source labels explicit. Output has one row per matched movie–director pair.
combined = tmdb.merge(
    imdb, on="imdb_id", how="inner",
    suffixes=("_tmdb", "_imdb"), validate="one_to_many",
)
print("TMDB Shape:", tmdb.shape)
print("IMDb Shape:", imdb.shape)
print("combined shapes:", combined.shape)
print("Matched unique movies:", combined["tmdb_id"].nunique())
print("^Because The inner join drops movies without an IMDb match and \na movie with two directors gets two rows, one for each director.")
combined.head()


TMDB Shape: (13990, 15)
IMDb Shape: (763387, 4)
combined shapes: (14280, 18)
Matched unique movies: 12777
^Because The inner join drops movies without an IMDb match and 
a movie with two directors gets two rows, one for each director.


,tmdb_id,imdb_id,title_tmdb,release_date,budget,revenue,runtime,genres,original_language,popularity,production_companies,director_tmdb,top_cast,vote_count,vote_average,nconst,director_imdb,title_imdb
0,769,tt0099685,GoodFellas,1990-09-12,25000000,47072327,145,Drama|Crime,en,24.8484,Winkler Films|Warner Bros. Pictures,Martin Scorsese,Robert De Niro|Ray Liotta|Joe Pesci|Lorraine B...,14873,8.449,nm0000217,Martin Scorsese,GoodFellas
1,162,tt0099487,Edward Scissorhands,1990-12-07,20000000,86024005,105,Fantasy|Drama|Romance,en,26.1281,20th Century Fox,Tim Burton,Johnny Depp|Winona Ryder|Dianne Wiest|Anthony ...,14109,7.718,nm0000318,Tim Burton,Edward Scissorhands
2,771,tt0099785,Home Alone,1990-11-16,18000000,476684675,103,Comedy|Family,en,1.4513,Hughes Entertainment|20th Century Fox,Chris Columbus,Macaulay Culkin|Joe Pesci|Daniel Stern|John He...,12941,7.500,nm0001060,Chris Columbus,Home Alone
3,196,tt0099088,Back to the Future Part III,1990-05-25,40000000,245077583,119,Adventure|Comedy|Science Fiction,en,11.9057,Universal Pictures|Amblin Entertainment,Robert Zemeckis,Michael J. Fox|Christopher Lloyd|Mary Steenbur...,11911,7.500,nm0000709,Robert Zemeckis,Back to the Future Part III
4,114,tt0100405,Pretty Woman,1990-03-23,14000000,463406268,120,Romance|Comedy,en,22.6004,Touchstone Pictures|Silver Screen Partners IV|...,Garry Marshall,Julia Roberts|Richard Gere|Ralph Bellamy|Jason...,9243,7.440,nm0005190,Garry Marshall,Pretty Woman


In [8]:
# Inspect Combined Column Names
print(combined.columns.tolist())

['tmdb_id', 'imdb_id', 'title_tmdb', 'release_date', 'budget', 'revenue', 'runtime', 'genres', 'original_language', 'popularity', 'production_companies', 'director_tmdb', 'top_cast', 'vote_count', 'vote_average', 'nconst', 'director_imdb', 'title_imdb']


In [9]:
# Compare Titles Across the Two Sources
print((combined["title_tmdb"] == combined["title_imdb"]).mean())

0.920798319327731


having a result of 1.0 means the titles were successfully merged

In [10]:
## Save the Combined Dataset to CSV
combined.to_csv("combined_dataset.csv", index=False)